In [ ]:
# Step 1 - Mount Drive correctly
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import spacy

gpu_available = spacy.prefer_gpu()
print(f"Is GPU available for spaCy? {gpu_available}")

Is GPU available for spaCy? True


In [ ]:
import re
import spacy
import unicodedata
import pandas as pd
from typing import List, Union

class TextCleaner:
    def __init__(self, use_lemmatization: bool = True):
        self.use_lemmatization = use_lemmatization
        if self.use_lemmatization:
            spacy.prefer_gpu()
            self.nlp = spacy.load(
                'en_core_web_sm',
                disable=['ner', 'parser', 'senter']
            )

    def clean_basic_structures(self, text: str):
        if not isinstance(text, str):
            return ""
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'<[^>]*>', '', text)
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _pre_clean(self, text: str):
        cleaned = self.clean_basic_structures(text).lower()
        cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', cleaned)
        return cleaned

    def pipeline(self, texts: Union[str, List[str], pd.Series], batch_size: int = 1024 ):
        if isinstance(texts, str):
            if self.use_lemmatization:
                doc = self.nlp(self._pre_clean(texts))
                tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]
                return ' '.join(tokens)
            else:
                return self.clean_basic_structures(texts)

        if isinstance(texts, pd.Series):
            texts = texts.tolist()

        if self.use_lemmatization:
            pre_cleaned = [self._pre_clean(t) for t in texts]
            results = []

            for doc in self.nlp.pipe(pre_cleaned, batch_size=batch_size, n_process=1):
                tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]
                results.append(' '.join(tokens))
            return results
        else:
            return [self.clean_basic_structures(t) for t in texts]

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def prepare_raw_isot(isot_true_path: str, isot_fake_path: str):
    isot_true_df = pd.read_csv(isot_true_path)
    isot_true_df['label'] = 1
    isot_fake_df = pd.read_csv(isot_fake_path)
    isot_fake_df['label'] = 0

    df_isot_combined = pd.concat([isot_true_df, isot_fake_df], ignore_index=True)
    df_isot_combined = df_isot_combined[['text', 'label']]
    # Shuffling 5 times using a loop with varying random seeds
    for i in range(5):
        # Generates a new seed for each pass (e.g., 42, 43, 44, 45, 46)
        current_seed = 42 + i
        df_isot_combined = df_isot_combined.sample(frac=1, random_state=current_seed).reset_index(drop=True)
    df_isot_combined.dropna(inplace=True)
    df_isot_combined.drop_duplicates(inplace=True)
    df_isot_combined = df_isot_combined.reset_index(drop=True)
    return df_isot_combined


def load_and_merge_datasets(wakfake_pth:str, isot_dataset: pd.DataFrame = None):
    logger.info(f'Loading WAKFake dataset...')
    wf_df = pd.read_csv(wakfake_pth)
    wf_df = wf_df[['text', 'label']].dropna().reset_index(drop=True)

    # Fix — assign it back
    wf_df['label'] = wf_df['label'].map({0: 1, 1: 0})

    if isot_dataset is not None:
        logger.info(f'Combining WAKFake and ISOT datasets into one dataset...')
        df_combined = pd.concat([wf_df, isot_dataset], ignore_index=True)
        df_combined.drop_duplicates(inplace=True)
        df_combined = df_combined.reset_index(drop=True)
        return df_combined

    return wf_df

def create_stratified_splits(
        df: pd.DataFrame,
        target_col: str = 'label',
        test_size: float = 0.15,
        val_size: float = 0.15
):
    # Creating a strict 70:15:15 stratified train/val/test split to preserve class distribution.
    logger.info("Executing stratified train/test/val splits...")

    combined_test_size = test_size + val_size
    train_df, temp_df = train_test_split(
        df,
        test_size=combined_test_size,
        stratify=df[target_col],
        random_state=42
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df[target_col],
        random_state=42
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    logger.info(f"Split completed. Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    return train_df, val_df, test_df

In [ ]:
import os
import pandas as pd

# 1. Define paths
project_path = '/content/drive/MyDrive/NLP_project'

# 2. Instantiate your cleaner
cleaner = TextCleaner(use_lemmatization=True)

# 3. Find and process all CSV files
for root, dirs, files in os.walk(project_path):
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)
            print(f"Processing: {file_path}")

Processing: /content/drive/MyDrive/NLP_project/ISOT_Fake.csv
Processing: /content/drive/MyDrive/NLP_project/ISOT_True.csv
Processing: /content/drive/MyDrive/NLP_project/WELFake.csv
Processing: /content/drive/MyDrive/NLP_project/processed_train_df.csv
Processing: /content/drive/MyDrive/NLP_project/processed_val_df.csv


In [ ]:
def run_baseline_pipeline(
        welfake_path: str,
        isot_true_path: str = None,
        isot_fake_path: str = None
):
    # Load, merge and shuffle datasets
    isot_df = None
    if isot_true_path is not None and isot_fake_path is not None:
        logger.info("Raw ISOT paths detected. Triggering ISOT loading and shuffling process...")
        isot_df = prepare_raw_isot(isot_true_path, isot_fake_path)
    else:
        logger.info("ISOT paths not provided. Pipeline will proceed using only the primary dataset.")

    master_df = load_and_merge_datasets(welfake_path, isot_df)

    # Creating Stratified Splits (70/15/15)
    logger.info('Creating Stratified splits of master_df')
    train_df, val_df, test_df = create_stratified_splits(master_df)

    # Text Preprocessing
    logger.info('Starting text preprocessing pipeline')
    cleaner = TextCleaner(use_lemmatization=True)

    train_df['cleaned_text'] = cleaner.pipeline(train_df['text'].tolist(), batch_size=1024)
    val_df['cleaned_text'] = cleaner.pipeline(val_df['text'].tolist(), batch_size=1024)
    test_df['cleaned_text'] = cleaner.pipeline(test_df['text'].tolist(), batch_size=1024)
    train_df = train_df[['text', 'label', 'cleaned_text']]
    val_df = val_df[['text', 'label', 'cleaned_text']]
    test_df = test_df[['text', 'label', 'cleaned_text']]

    # Save as CSV
    train_df.to_csv('/content/drive/MyDrive/NLP_project/processed_train_df.csv', index=False, encoding='utf-8')

    val_df.to_csv('/content/drive/MyDrive/NLP_project/processed_val_df.csv', index=False, encoding='utf-8')

    test_df.to_csv('/content/drive/MyDrive/NLP_project/processed_test_df.csv', index=False, encoding='utf-8')

In [ ]:
run_baseline_pipeline(welfake_path='/content/drive/MyDrive/NLP_project/WELFake.csv',
                      isot_true_path='/content/drive/MyDrive/NLP_project/ISOT_True.csv',
                      isot_fake_path='/content/drive/MyDrive/NLP_project/ISOT_Fake.csv')